# HW03 — Model Serving & Deployment

Your model is trained and tracked in MLflow. A model that only runs in a notebook is not useful in production.

In this homework you will:

- verify that your trained model is reproducible and ready for serving
- wrap it in a FastAPI service with proper input validation and batch support
- measure the performance difference between single and batch prediction
- package the service in Docker with a size-optimized image
- write Kubernetes manifests to deploy the service

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords, tokens, or notebook checkpoints.
Do not hardcode passwords anywhere in your code.

## Useful references

- MLflow model loading: https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html
- FastAPI: https://fastapi.tiangolo.com/
- FastAPI lifespan: https://fastapi.tiangolo.com/advanced/events/
- Pydantic v2: https://docs.pydantic.dev/latest/
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker multi-stage builds: https://docs.docker.com/build/building/multi-stage/
- Kubernetes Deployments: https://kubernetes.io/docs/concepts/workloads/controllers/deployment/
- Kubernetes Services: https://kubernetes.io/docs/concepts/services-networking/service/

## What to avoid

- Loading the model inside the request handler. Load once at startup.
- Hardcoded passwords in any source file.
- A Docker image that bakes in the model file. Pull from MLflow at startup.
- Returning raw numpy types from the API. JSON needs native Python types.
- Skipping the batch vs single benchmark. The numbers tell a story.

In [ ]:
import os
import re
import time
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

PROJECT = Path.cwd()
for path in ['src/airbnb_serving', 'k8s', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_serving/__init__.py').write_text('__version__ = "0.1.0"\n')

STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or input('GitHub username / student id: ').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
EXPERIMENT_NAME = f'qbc12_hw02_{safe_student}'

print('PROJECT:', PROJECT)
print('EXPERIMENT_NAME:', EXPERIMENT_NAME)

---
## Part 1 — Model Reproducibility Check

Before serving a model, you must verify it produces exactly the same output as it did during training.

This is called a **reproducibility check** and it catches silent bugs like:
- preprocessing mismatch between training and serving
- wrong model version loaded
- feature column order changed

### 1.1 Connect to MLflow and load your best model

In [2]:
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://185.50.38.163:33014')
MLFLOW_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME', '') or input('MLflow username: ').strip()
MLFLOW_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD', '') or input('MLflow password: ').strip()

os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f'Experiment not found: {EXPERIMENT_NAME}. Complete HW02 first.')

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.leakage_status = 'clean' and tags.selected_for_serving = 'true'",
    order_by=['metrics.f1 DESC'],
)

if runs.empty:
    raise ValueError(
        'No run tagged selected_for_serving=true found. '
        'Go to MLflow UI, find your best clean run, and add the tag.'
    )

BEST_RUN_ID = runs.iloc[0]['run_id']
MODEL_URI = f'runs:/{BEST_RUN_ID}/model'
os.environ['MODEL_URI'] = MODEL_URI

MODEL_RUN_ID = BEST_RUN_ID
os.environ['MODEL_RUN_ID'] = MODEL_RUN_ID


print('Best run ID  :', BEST_RUN_ID)
print('Model URI    :', MODEL_URI)
print('Run name     :', runs.iloc[0].get('tags.mlflow.runName'))
print('F1 score     :', runs.iloc[0].get('metrics.f1'))
print(os.getenv('MODEL_URI'))
print(os.getenv('MODEL_RUN_ID'))



Best run ID  : b03526c7c67540f9a5566c3277b0f150
Model URI    : runs:/b03526c7c67540f9a5566c3277b0f150/model
Run name     : v5_random_forest1
F1 score     : 0.9300911854103343
runs:/b03526c7c67540f9a5566c3277b0f150/model
b03526c7c67540f9a5566c3277b0f150


In [3]:
model = mlflow.sklearn.load_model(MODEL_URI)
print('Model type:', type(model))
print('Model steps:', list(model.named_steps.keys()) if hasattr(model, 'named_steps') else 'not a pipeline')

Model type: <class 'sklearn.pipeline.Pipeline'>
Model steps: ['preprocess', 'model']


### 1.2 Load your HW01 feature dataset

You will use a small sample from your HW01 feature parquet file to verify reproducibility.

In [4]:
FEATURE_COLS = [
    'room_type', 'property_type', 'neighbourhood_name',
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price',
    'minimum_nights', 'maximum_nights', 'instant_bookable', 'host_is_superhost',
    'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff',
    'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff',
    'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d',
    'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d',
    'available_days_last_30d', 'available_rate_last_30d',
    'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d',
]
TARGET_COL = 'high_demand_proxy'

# Load your HW01 parquet file
# Adjust the path if needed
path = Path('C:/Users/lenovo/Desktop/hw2/1/data/features')
parquet_path = list((path).glob('*.parquet'))
if not parquet_path:
    raise FileNotFoundError('HW01 feature parquet not found. Run HW01 ETL first.')

df = pd.read_parquet(parquet_path[0])
print('Dataset shape:', df.shape)
df[FEATURE_COLS + [TARGET_COL]].head(3)

Dataset shape: (9383, 33)


,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,...,days_since_last_review,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,high_demand_proxy
0,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,132.0,3,356,...,116,17,0.188889,3.000000,30.0,1,0.033333,3.000000,30.0,1.0
1,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,0.2,89.0,2,730,...,116,15,0.166667,1.888889,730.0,6,0.200000,1.933333,730.0,1.0
2,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,0.2,61.0,2,730,...,115,13,0.144444,1.977778,730.0,2,0.066667,2.000000,730.0,1.0


### 1.3 Reproducibility check

**TODO 1.3**

Take a sample of 50 rows from the dataset.

Run `model.predict()` on those rows **twice** and verify the results are identical.

Then compare the predictions against the `high_demand_proxy` column and print:
- how many predictions match the training label
- the accuracy on this sample

If both runs produce identical output, print `Reproducibility check passed.`
If they differ, raise a `ValueError`.

In [5]:
# TODO 1.3
# Write your reproducibility check here.

sample = df[FEATURE_COLS + [TARGET_COL]].sample(50, random_state=42)
X_sample = sample[FEATURE_COLS]
y_sample = sample[TARGET_COL]

# Your code here

In [6]:
y_pred1 = model.predict(X_sample)
matches = (y_pred1 == y_sample).sum()
accuracy = matches / len(y_sample)
accuracy



np.float64(0.98)

In [7]:
y_pred2 = model.predict(X_sample)
matches = (y_pred2 == y_sample).sum()
accuracy = matches / len(y_sample)
accuracy


np.float64(0.98)

In [8]:
if not np.array_equal(y_pred1, y_pred2):
    raise ValueError
else:
    print("Reproducibility check passed.")

Reproducibility check passed.


---
## Part 2 — FastAPI Service

A REST API is the standard way to expose an ML model to other systems.

You will build a FastAPI app with two prediction endpoints:
- `POST /predict` — single listing prediction
- `POST /predict/batch` — multiple listings in one request

Then you will measure how much faster batch is compared to calling single predict in a loop.

### 2.1 Input and output schemas

In [9]:
for x in df.columns:
    if df[x].hasnans:
        print(x)

host_is_superhost
bedrooms
beds
bathrooms
listing_price


In [24]:
# TODO 2.1
#Create src/airbnb_serving/schema.py
# Define two Pydantic models:
#
# ListingFeatures:
#   - all feature columns from FEATURE_COLS above
#   - use correct types: str, int, float, bool
#   - nullable fields (those with NaN in dataset) should use Optional[float] = None

   
# PredictionResponse:
#   - listing_id: int | None = None
#   - prediction: int  (0 or 1)
#   - probability_high_demand: float
#   - model_run_id: str
#
 

(PROJECT / 'src/airbnb_serving/schema.py').write_text(textwrap.dedent('''
# Write your Pydantic schemas here

from pydantic import BaseModel, Field
from typing import Optional, Literal

class ListingFeatures(BaseModel):
    listing_id: int | None = None
    room_type : str
    property_type : str 
    neighbourhood_name: str

    accommodates : int = Field(..., ge = 0)  

    bedrooms : Optional[float] = Field(None, ge = 0)
    beds : Optional[float] = Field(None, ge = 0) 
    bathrooms: Optional[float] = Field(None, ge = 0)
    listing_price: Optional[float] = Field(None, ge = 0)

    minimum_nights: int = Field(..., ge = 0)
    maximum_nights: int = Field(..., ge = 0) 

    instant_bookable: bool 
    host_is_superhost: Optional[bool] = None

    host_listing_count: int = Field(..., ge = 0)

    total_reviews_before_cutoff: Optional[float] = Field(..., ge = 0)
    unique_reviewers_before_cutoff: Optional[float] = Field(..., ge = 0)
    avg_comment_len_before_cutoff : Optional[float] = Field(..., ge = 0) 
    max_comment_len_before_cutoff : Optional[float] = Field(..., ge = 0)
    days_since_last_review : Optional[float] = Field(..., ge = 0)

    available_days_last_90d: int = Field(..., ge = 0)

    available_rate_last_90d: float = Field(..., ge = 0, le = 1)

    avg_minimum_nights_calendar_last_90d: Optional[float] = Field(..., ge = 0) 
    avg_maximum_nights_calendar_last_90d: Optional[float] = Field(..., ge = 0)

    available_days_last_30d: int = Field(..., ge = 0)
    available_rate_last_30d: float = Field(..., ge = 0, le = 1)

    avg_minimum_nights_calendar_last_30d: Optional[float] = Field(..., ge = 0)
    avg_maximum_nights_calendar_last_30d: Optional[float] = Field(..., ge = 0)


class PredictionResponse(BaseModel):
    listing_id: int | None = None
    prediction: Literal[0, 1]
    probability_high_demand: float
    model_run_id: str



 ''').strip() + '\n')

1805

### 2.2 Prediction logic

In [19]:
# TODO 2.2
# Create src/airbnb_serving/predictor.py
#
# Add two functions:
#
# predict_single(features: ListingFeatures, model, run_id: str) -> PredictionResponse
#   - convert ListingFeatures to a single-row DataFrame
#   - call model.predict and model.predict_proba
#   - return PredictionResponse
#   - all values must be native Python types (int, float), not numpy types
#
# predict_batch(features_list: list[ListingFeatures], model, run_id: str) -> list[PredictionResponse]
#   - convert list to a multi-row DataFrame in one step
#   - call model.predict and model.predict_proba once for the whole batch
#   - return a list of PredictionResponse
#   - do NOT loop and call predict_single for each row



(PROJECT / 'src/airbnb_serving/predictor.py').write_text(textwrap.dedent('''



import pandas as pd
from .schema import ListingFeatures, PredictionResponse

def predict_single(features: ListingFeatures, model, run_id: str) -> PredictionResponse:
    x = pd.DataFrame([features.model_dump()])
    print(type(model))
    print(hasattr(model, "predict_proba"))
    pred = model.predict(x)[0]
    prob = model.predict_proba(x)[0, 1]
    
    
    return PredictionResponse(
                listing_id= features.listing_id,
                prediction= int(pred),
                probability_high_demand= float(prob),
                model_run_id= str(run_id)  
            )
    



def predict_batch(features_list: list[ListingFeatures], model, run_id: str) -> list[PredictionResponse]:
    rows = [ feature.model_dump() for feature in features_list]
    x = pd.DataFrame(rows)
    
    predictions = model.predict(x)
    probabilities = model.predict_proba(x)[:, 1]
    
    responses = []

    for feature, pred, prob in zip(features_list, predictions, probabilities):
        responses.append(
            PredictionResponse(
                listing_id= feature.listing_id,
                prediction= int(pred),
                probability_high_demand= float(prob),
                model_run_id= str(run_id)  
            )
        )
    
    return responses


 ''').strip() + '\n')

1256

### 2.3 FastAPI app

In [12]:
# TODO 2.3

# Create src/airbnb_serving/app.py
#
#   GET /health
#     response: {"status": "ok", "model_run_id": str}
#
#   POST /predict
#     request body: ListingFeatures
#     response: PredictionResponse
#
#   POST /predict/batch
#     request body: list[ListingFeatures]
#     response: list[PredictionResponse]
#
# Rules:
#   - Load the model ONCE using a lifespan context manager, not inside handlers
#   - Read MODEL_RUN_ID, MLFLOW_TRACKING_URI, MLFLOW_TRACKING_USERNAME,
#     MLFLOW_TRACKING_PASSWORD from environment variables
#   - Use the predictor module for prediction logic


(PROJECT / 'src/airbnb_serving/app.py').write_text(textwrap.dedent('''
from __future__ import annotations
from dotenv import load_dotenv
load_dotenv()
import os

import mlflow


from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException, status, Request

from .predictor import predict_single, predict_batch
from .schema import ListingFeatures, PredictionResponse

MODEL_RUN_ID = os.getenv("MODEL_RUN_ID")
MODEL_URI = os.getenv("MODEL_URI")
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")
print("MODEL_URI =", MODEL_URI)
print("MODEL_RUN_ID =", MODEL_RUN_ID)

@asynccontextmanager
async def lifespan(app:FastAPI):

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

    if MLFLOW_TRACKING_USERNAME and MLFLOW_TRACKING_PASSWORD:
        os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
        os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD
    global model
    model = mlflow.sklearn.load_model(MODEL_URI)
    app.state.model = model

    yield

app = FastAPI(lifespan=lifespan)


@app.get('/health')
def health():
    return {"status": "ok", 
            "model_run_id": str(MODEL_RUN_ID)}


@app.post("/predict")
def predict(payload: ListingFeatures, request: Request):
    model = request.app.state.model
    return predict_single(payload, model, MODEL_RUN_ID)

@app.post('/predict/batch')
def predictBatch(payload: list[ListingFeatures])-> list[PredictionResponse]:
    return predict_batch(payload, model, MODEL_RUN_ID)




 ''').strip() + '\n')

1553

### 2.4 Package metadata

In [13]:
# TODO 2.4
# Create pyproject.toml and requirements.txt
#
# Package name: airbnb-serving
# Source directory: src/
# Required dependencies:
#   fastapi>=0.111
#   uvicorn[standard]>=0.29
#   mlflow>=2.13
#   scikit-learn>=1.4
#   pandas>=2.0
#   pydantic>=2.0
#   python-dotenv>=1.0

(PROJECT / 'pyproject.toml').write_text(textwrap.dedent('''
[build-system]
requires = ["setuptools", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "airbnb-serving"
version = "0.1.0"
description = "ML serving layer"
requires-python = ">=3.9"
dependencies = [
    "fastapi>=0.111",
    "uvicorn[standard]>=0.29",
    "mlflow>=2.13",
    "scikit-learn>=1.4",
    "pandas>=2.0",
    "pydantic>=2.0",
    "python-dotenv>=1.0"
]

[tool.setuptools]
package-dir = {"" = "src"}

[tool.setuptools.packages.find]
where = ["src"]
''').strip() + '\n')


479

In [15]:
(PROJECT / 'src/requirements.txt').write_text(textwrap.dedent('''
fastapi>=0.111
uvicorn[standard]>=0.29
mlflow>=2.13
scikit-learn>=1.4
pandas>=2.0
pydantic>=2.0
python-dotenv>=1.0
''').strip() + '\n')

115

### 2.5 Local install and smoke test

Install the package and manually start the server in a terminal before running the test cell below.

```bash
pip install -e .

MODEL_RUN_ID= \
MLFLOW_TRACKING_URI= \
MLFLOW_TRACKING_USERNAME= \
MLFLOW_TRACKING_PASSWORD= \
uvicorn airbnb_serving.app:app --host 0.0.0.0 --port 8000
```

In [16]:
import sys
!{sys.executable} -m pip install -q -e .

In [ ]:
import subprocess, time, signal, os

server_proc = subprocess.Popen(
    ['uvicorn', 'src.airbnb_serving.app:app', '--host', '0.0.0.0', '--port', '12345'],
    env={**os.environ, 'MODEL_RUN_ID': BEST_RUN_ID,
         "MODEL_URI": f"runs:/{BEST_RUN_ID}/model"},
)
time.sleep(15)  # wait for model to load from MLflow
print('Server started, PID:', server_proc.pid)
print(MODEL_RUN_ID)

Server started, PID: 15180
b03526c7c67540f9a5566c3277b0f150


In [20]:
import requests

BASE_URL = 'http://localhost:12345'

health = requests.get(f'{BASE_URL}/health')
print(health.status_code )
assert health.status_code == 200, f'Health check failed: {health.text}'
print('Health:', health.json())

sample_payload = {
    'room_type': 'Entire home/apt',
    'property_type': 'Entire rental unit',
    'neighbourhood_name': 'Centrum-West',
    'accommodates': 2,
    'bedrooms': 1.0,
    'beds': 1.0,
    'bathrooms': 1.0,
    'listing_price': 150.0,
    'minimum_nights': 2,
    'maximum_nights': 365,
    'instant_bookable': True,
    'host_is_superhost': False,
    'host_listing_count': 1,
    'total_reviews_before_cutoff': 10.0,
    'unique_reviewers_before_cutoff': 9.0,
    'avg_comment_len_before_cutoff': 120.0,
    'max_comment_len_before_cutoff': 300.0,
    'days_since_last_review': 30.0,
    'available_days_last_90d': 45,
    'available_rate_last_90d': 0.5,
    'avg_minimum_nights_calendar_last_90d': 2.0,
    'avg_maximum_nights_calendar_last_90d': 365.0,
    'available_days_last_30d': 15,
    'available_rate_last_30d': 0.5,
    'avg_minimum_nights_calendar_last_30d': 2.0,
    'avg_maximum_nights_calendar_last_30d': 365.0,
}

resp = requests.post(f'{BASE_URL}/predict', json=sample_payload)

print(resp.status_code)
print(resp.text)
assert resp.status_code == 200, f'Single predict failed: {resp.text}'
print('Single predict:', resp.json())

print('Local smoke test passed.')

200
Health: {'status': 'ok', 'model_run_id': 'b03526c7c67540f9a5566c3277b0f150'}
200
{"listing_id":null,"prediction":0,"probability_high_demand":0.36,"model_run_id":"b03526c7c67540f9a5566c3277b0f150"}
Single predict: {'listing_id': None, 'prediction': 0, 'probability_high_demand': 0.36, 'model_run_id': 'b03526c7c67540f9a5566c3277b0f150'}
Local smoke test passed.


### 2.6 Batch vs single benchmark

**TODO 2.6**

Take 100 rows from your feature dataset.

Measure:
1. Time to call `POST /predict` 100 times in a loop (single)
2. Time to call `POST /predict/batch` once with all 100 rows (batch)

Print a comparison table with total time and time per prediction for each approach.

Then answer: why is batch faster? Write your answer as a comment in the cell.

In [ ]:
# TODO 2.6
# Write your benchmark here.
# Hint: convert df rows to list of dicts with df.to_dict(orient='records')

import time
import requests
import pandas as pd
import math

def clean_for_json(row: dict) -> dict:
    return {
        k: None if isinstance(v, float) and math.isnan(v) else v
        for k, v in row.items()
    }

BENCHMARK_SIZE = 100
benchmark_rows = [
    clean_for_json(row)
    for row in df[FEATURE_COLS].head(BENCHMARK_SIZE).to_dict(orient='records')
]

# Your benchmark code here

# Single pred loop

start = time.perf_counter()


for row in benchmark_rows:
    resp = requests.post(
        f"{BASE_URL}/predict",
        json=row
    )
    # assert resp.status_code == 200


single_total = time.perf_counter() - start

# Batch pred

start = time.perf_counter()

resp = requests.post(
    f"{BASE_URL}/predict/batch",
    json=benchmark_rows
)
assert resp.status_code == 200

batch_total = time.perf_counter() - start

# --- Print results ---
# Example output format:
# Method        | Total (s) | Per prediction (ms)
# single loop   |   X.XX    |       XX.X
# batch         |   X.XX    |        X.X
# Speedup: Xx

single_ms = single_total * 1000 / BENCHMARK_SIZE
batch_ms = batch_total * 1000 / BENCHMARK_SIZE
speedup = single_total / batch_total

print(f"{'Method':<15} | {'Total (s)':<10} | {'Per prediction (ms)'}")
print("-" * 50)
print(f"{'single loop':<15} | {single_total:<10.3f} | {single_ms:<.2f}")
print(f"{'batch':<15} | {batch_total:<10.3f} | {batch_ms:<.2f}")
print("-" * 50)
print(f"Speedup: {speedup:.2f}x")

# --- Answer ---
# Why is batch faster? (write as a comment below)
# ...

Method          | Total (s)  | Per prediction (ms)
--------------------------------------------------
single loop     | 211.287    | 2112.87
batch           | 2.159      | 21.59
--------------------------------------------------
Speedup: 97.88x


In [26]:
server_proc.terminate()
print('Server stopped.')

Server stopped.


---
## Part 3 — Docker

You will build two Docker images and compare their sizes.

This teaches you that image size is not free — it affects pull time, storage cost, and attack surface.

### 3.1 Naive Dockerfile

In [27]:
# TODO 3.1
# Create Dockerfile.naive
#
# A simple, unoptimized Dockerfile:
#   FROM python:3.11
#   WORKDIR /app
#   COPY . .
#   RUN pip install -r requirements.txt && pip install -e .
#   EXPOSE 8000
#   CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]

# Write your code here.

In [61]:
(PROJECT / "Dockerfile.naive").write_text(
"""FROM python:3.11

WORKDIR /app

COPY . .

RUN pip install -r requirements.txt 

EXPOSE 8000

CMD ["uvicorn", "src.airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]
"""
)

178

### 3.2 Optimized Dockerfile

A multi-stage build separates the build environment from the runtime environment.

Stage 1 (builder): install everything, build the package.
Stage 2 (runtime): copy only what is needed to run, nothing else.

In [30]:
# TODO 3.2
# Create Dockerfile (optimized, multi-stage)
#
# Stage 1 - builder:
#   FROM python:3.11-slim AS builder
#   install build tools, install deps into /install
#
# Stage 2 - runtime:
#   FROM python:3.11-slim
#   copy only /install from builder
#   copy src/
#   EXPOSE 8000
#   CMD uvicorn ...
#
# Also create .dockerignore to exclude:
#   .git, .venv, __pycache__, .ipynb_checkpoints, *.pyc, data/, .env

# Write your code here.

In [80]:
(PROJECT / "Dockerfile").write_text(
"""FROM python:3.11-slim AS builder
WORKDIR /app

COPY requirements.txt .

RUN pip install --upgrade pip && \
    pip install --target=/install -r requirements.txt

FROM python:3.11-slim

WORKDIR /app

ENV PYTHONPATH=/app:/install

COPY --from=builder /install /install
COPY src/ src/
COPY pyproject.toml .

EXPOSE 8000

CMD ["python", "-m", "uvicorn", "src.airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]"""
)

(PROJECT / ".dockerignore").write_text(
""".git
.venv
__pycache__
.ipynb_checkpoints
*.pyc
data/
.env
"""
)

59

### 3.3 Build and compare image sizes

In [50]:
!docker build -f Dockerfile.naive -t qbc12-airbnb-serving:naive .
!docker build -f Dockerfile -t qbc12-airbnb-serving:optimized .

#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.naive
#1 transferring dockerfile: 230B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11
#2 DONE 0.3s

#3 [1/4] FROM docker.io/library/python:3.11@sha256:a30c4ff1a6a474019f9b1f0d921e81a254cf420d408c09e8a8b79fd803b62ebf
#3 resolve docker.io/library/python:3.11@sha256:a30c4ff1a6a474019f9b1f0d921e81a254cf420d408c09e8a8b79fd803b62ebf 0.0s done
#3 DONE 0.0s

#4 [2/4] WORKDIR /app
#4 CACHED

#5 [internal] load .dockerignore
#5 transferring context: 106B done
#5 DONE 0.0s

#6 [internal] load build context
#6 transferring context: 157.81kB 0.0s done
#6 DONE 0.0s

#7 [3/4] COPY . .
#7 DONE 0.1s

#8 [4/4] RUN pip install -r requirements.txt
#8 2.879 Collecting fastapi>=0.111 (from -r requirements.txt (line 1))
#8 3.804   Downloading fastapi-0.137.2-py3-none-any.whl.metadata (27 kB)
#8 4.170 Collecting uvicorn>=0.29 (from uvicorn[standard]>=0.29->-r req

In [76]:
# TODO 3.3
# Run this cell after building both images.
# It compares image sizes and saves the result to reports/.

import subprocess, json

result = subprocess.run(
    ['docker', 'images', '--format', '{{json .}}'],
    capture_output=True, text=True
)

images = [json.loads(line) for line in result.stdout.strip().split('\n') if line]
serving_images = [
    img for img in images
    if img.get('Repository') == 'qbc12-airbnb-serving'
]

size_df = pd.DataFrame(serving_images)[['Repository', 'Tag', 'Size']]
print(size_df.to_string(index=False))

report_lines = [
    '# HW03 Docker Image Size Report', '',
    size_df.to_markdown(index=False), '',
    '## Analysis',
    '<!-- TODO: Write 2-3 sentences explaining why the sizes differ and which you would use in production. -->',
    """I prefer to use the optimized image  since it is significantly smaller than the naive image. It uses a slimmer base image and 
    avoids copying unnecessary files into the container. A smaller image reduces storage requirements, speeds up image downloads, and improves deployment times.
    For production use, I would choose the optimized image because it is more efficient, faster to deploy, and generally has a smaller attack surface."""
]

Path('reports/docker_size_report.md').write_text('\n'.join(report_lines) + '\n')
print('\nReport saved to reports/docker_size_report.md')

          Repository       Tag   Size
qbc12-airbnb-serving optimized 1.29GB
qbc12-airbnb-serving     naive 3.13GB

Report saved to reports/docker_size_report.md


### 3.4 Docker Compose

In [ ]:
# TODO 3.4
# Create docker-compose.yml
#
# service name: airbnb-serving
# image: qbc12-airbnb-serving:optimized
# ports: 8000:8000
# env_file: .env
#
# Also create .env.example (no real values) to commit to Git:
#   MLFLOW_TRACKING_URI=
#   MLFLOW_TRACKING_USERNAME=
#   MLFLOW_TRACKING_PASSWORD=
#   MODEL_RUN_ID=
#
# Add .env to .gitignore

# Write your code here.

In [77]:
(PROJECT / "docker-compose.yml").write_text(
"""services:
  airbnb-serving:
    build:
      context: .
      dockerfile: Dockerfile
    image: qbc12-airbnb-serving:optimized
    ports:
      - "8000:8000"
    env_file:
      - .env
"""
)

(PROJECT / ".env.example").write_text(
"""MLFLOW_TRACKING_URI=
MLFLOW_TRACKING_USERNAME=
MLFLOW_TRACKING_PASSWORD=
MODEL_RUN_ID=
MODEL_URI=
"""
)

print("Created docker-compose.yml")
print("Created .env.example")

# with open(PROJECT / ".gitignore", "a") as f:
#     f.write("\n.env\n")

gitignore_path = PROJECT / ".gitignore"
content = gitignore_path.read_text() if gitignore_path.exists() else ""

if ".env" not in content:
    gitignore_path.write_text(content + "\n.env\n")

print("Updated .gitignore")

Created docker-compose.yml
Created .env.example
Updated .gitignore


In [82]:
# !type docker-compose.yml
# !type .env.example

In [83]:
# !docker compose down
# !docker build --no-cache -t qbc12-airbnb-serving:optimized .
# !docker compose up

In [81]:
# Docker Compose smoke test
!docker compose up -d

import time, requests
time.sleep(8)  # wait for model to load from MLflow

health = requests.get('http://localhost:8000/health')
assert health.status_code == 200, f'Failed: {health.text}'
print('Docker Compose health check passed:', health.json())

!docker compose down

 Container hw3-airbnb-serving-1 Running 


Docker Compose health check passed: {'status': 'ok', 'model_run_id': 'b03526c7c67540f9a5566c3277b0f150'}


 Container hw3-airbnb-serving-1 Stopping 
 Container hw3-airbnb-serving-1 Stopped 
 Container hw3-airbnb-serving-1 Removing 
 Container hw3-airbnb-serving-1 Removed 
 Network hw3_default Removing 
 Network hw3_default Removed 


---
## Part 4 — Kubernetes Manifests

Kubernetes is the standard way to run containers in production at scale.

You do not need a real cluster for this homework. The deliverable is correct YAML files that a cluster could apply.

Key concepts you will use:

| Concept | What it does |
|---|---|
| **Pod** | Runs your container |
| **Deployment** | Manages multiple identical Pods, handles restarts |
| **Service** | Stable network endpoint that routes traffic to Pods |
| **Secret** | Stores sensitive values like passwords, not plaintext in YAML |
| **readinessProbe** | Tells Kubernetes when a Pod is ready to receive traffic |
| **resource limits** | Prevents one Pod from consuming all server memory |

### 4.1 Deployment

In [ ]:
# TODO 4.1
# Create k8s/deployment.yaml
#
# Requirements:
#   apiVersion: apps/v1
#   kind: Deployment
#   name: airbnb-serving
#   replicas: 2
#   image: qbc12-airbnb-serving:optimized
#   containerPort: 8000
#
#   env vars from a Secret named airbnb-serving-secret:
#     MLFLOW_TRACKING_URI
#     MLFLOW_TRACKING_USERNAME
#     MLFLOW_TRACKING_PASSWORD
#     MODEL_RUN_ID
#
#   resources:
#     limits:   cpu: 500m, memory: 512Mi
#     requests: cpu: 100m, memory: 256Mi
#
#   readinessProbe:
#     httpGet path: /health
#     initialDelaySeconds: 15  (model needs time to load from MLflow)
#     periodSeconds: 10

# Write your code here.

In [84]:
%%writefile k8s/deployment.yaml

apiVersion: apps/v1
kind: Deployment
metadata:
  name: airbnb-serving
spec:
  replicas: 2
  selector:
    matchLabels:
      app: airbnb-serving
  template:
    metadata:
      labels:
        app: airbnb-serving
    spec:
      containers:
        - name: airbnb-serving
          image: qbc12-airbnb-serving:optimized
          ports:
            - containerPort: 8000

          env:
            - name: MLFLOW_TRACKING_URI
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MLFLOW_TRACKING_URI

            - name: MLFLOW_TRACKING_USERNAME
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MLFLOW_TRACKING_USERNAME

            - name: MLFLOW_TRACKING_PASSWORD
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MLFLOW_TRACKING_PASSWORD

            - name: MODEL_RUN_ID
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MODEL_RUN_ID

          resources:
            requests:
              cpu: "100m"
              memory: "256Mi"
            limits:
              cpu: "500m"
              memory: "512Mi"

          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 15
            periodSeconds: 10

Writing k8s/deployment.yaml


In [88]:
%%writefile k8s/secret.yaml
apiVersion: v1
kind: Secret
metadata:
  name: airbnb-serving-secret
type: Opaque
stringData:
  MLFLOW_TRACKING_URI: "your_uri"
  MLFLOW_TRACKING_USERNAME: "your_username"
  MLFLOW_TRACKING_PASSWORD: "your_password"
  MODEL_RUN_ID: "b03526c7c67540f9a5566c3277b0f150"

Writing k8s/secret.yaml


### 4.2 Service

In [ ]:
# TODO 4.2
# Create k8s/service.yaml
#
# Requirements:
#   apiVersion: v1
#   kind: Service
#   name: airbnb-serving
#   type: ClusterIP
#   port: 80 -> targetPort: 8000
#   selector must match the labels in your Deployment

# Write your code here.

In [89]:
%%writefile  k8s/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: airbnb-serving
spec:
  type: ClusterIP
  selector:
    app: airbnb-serving
  ports:
    - port: 80
      targetPort: 8000
      protocol: TCP

Writing k8s/service.yaml


### 4.3 Conceptual questions

Answer these in the markdown cell below. One or two sentences each is enough.

**TODO 4.3 — Answer here:**

**Q1.** We set `replicas: 2` instead of 1. What happens to traffic if one Pod crashes while replicas is 1 vs 2?

A: if replicas == 1 and the pod crashes, kubernetes will eventually restart a new pod. all requests during the gap will fail. 
if replicas == 2 and one pod crashes, load-balancing happens and Kubernetes removes the pod from the Service endpoints and Remaining Pod continues serving traffic
A new Pod is started to restore desired state

**Q2.** The `readinessProbe` has `initialDelaySeconds: 15`. Why do we need a delay specifically for this service?

A: the service is not instantly ready after the container starts. So Kubernetes waits for model load to completely and avoids false negatives and
only routes traffic once the service is actually usable

**Q3.** Why do we store credentials in a Kubernetes Secret instead of writing them directly in `deployment.yaml`?

A: It it not a good practice to do so. plus avoids the exposure of credetials in Git. Safety and Access control. COnfig, Code separation. easy updating

**Q4.** What is the difference between `ClusterIP` and `LoadBalancer` service types? When would you use each?

A: ClusterIP is only reachable by other Pods inside Kubernetes. used in backend services, internal microservices communications and avoided public exposure
LoadBalancer is accessible from outside the cluster & Cloud provider assigns a public IP. used for public APIs, production web services, SaaS endpoints, direct external access without port-forwarding

---
## Final Proof

If this cell fails, HW03 is not complete.

In [90]:
required_files = [
    'src/airbnb_serving/__init__.py',
    'src/airbnb_serving/schema.py',
    'src/airbnb_serving/predictor.py',
    'src/airbnb_serving/app.py',
    'pyproject.toml',
    'requirements.txt',
    'Dockerfile',
    'Dockerfile.naive',
    'docker-compose.yml',
    '.env.example',
    '.dockerignore',
    'k8s/deployment.yaml',
    'k8s/service.yaml',
    'reports/docker_size_report.md',
]

missing = [f for f in required_files if not (PROJECT / f).exists()]
assert not missing, f'Missing files:\n' + '\n'.join(missing)

# Check .env is gitignored
gitignore = (PROJECT / '.gitignore').read_text() if (PROJECT / '.gitignore').exists() else ''
assert '.env' in gitignore, '.env must be in .gitignore'

# Check Dockerfile does not copy .env
for df_name in ['Dockerfile', 'Dockerfile.naive']:
    content = (PROJECT / df_name).read_text()
    assert 'COPY .env' not in content, f'Do not copy .env in {df_name}'

# Check schema.py has actual content
schema_content = (PROJECT / 'src/airbnb_serving/schema.py').read_text()
assert 'BaseModel' in schema_content, 'schema.py must define Pydantic models'

# Check app.py has endpoints
app_content = (PROJECT / 'src/airbnb_serving/app.py').read_text()
assert '/health' in app_content, 'app.py must have /health endpoint'
assert '/predict' in app_content, 'app.py must have /predict endpoint'
assert 'batch' in app_content, 'app.py must have /predict/batch endpoint'

print('All required files present.')
print('No credential leaks detected.')
print('HW03 proof passed.')

All required files present.
No credential leaks detected.
HW03 proof passed.


## Screenshots required

Add these to the `screenshots/` folder before submitting:

- `screenshots/health_endpoint.png` — GET /health response
- `screenshots/predict_endpoint.png` — POST /predict response
- `screenshots/batch_endpoint.png` — POST /predict/batch response
- `screenshots/fastapi_docs.png` — auto-generated docs at /docs
- `screenshots/docker_image_sizes.png` — output of `docker images` showing both image sizes

## Commit

```bash
git add .
git commit -m "HW03 model serving and deployment"
git push
```